In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "6c709224",
   "metadata": {},
   "source": [
    "# 🎬 Module 6: Item-Based Collaborative Filtering\n",
    "**Movie Recommendation System Using Machine Learning**\n",
    "\n",
    "---\n",
    "\n",
    "### 🎯 Objectives\n",
    "1. Understand the intuition and mathematical formula for **Item-to-Item Cosine Similarity**.\n",
    "2. Compute cosine similarity efficiently using transposed SciPy sparse matrices.\n",
    "3. Find movies most similar to any selected movie.\n",
    "4. Predict user ratings for unseen movies:\n",
    "   $$\\hat{r}_{u,i} = \\frac{\\sum_{j \\in R(u)} s_{i,j} \\cdot r_{u,j}}{\\sum_{j \\in R(u)} |s_{i,j}|}$$\n",
    "5. Exclude already-watched movies and rank the top-K recommendations.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "ed5db448",
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import sys\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "\n",
    "sys.path.insert(0, \"..\")\n",
    "from src.data_loader import load_dataset\n",
    "from src.preprocessing import clean_movies, clean_ratings, filter_insufficient_ratings, train_test_split_ratings, build_user_item_matrix\n",
    "from src.similarity import compute_item_similarity, get_top_similar_items\n",
    "from src.recommender import MovieRecommender\n",
    "\n",
    "print(\"Imports ready!\")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "51bf07c1",
   "metadata": {},
   "source": [
    "## 1. Mathematical Concept: Item-Item Cosine Similarity\n",
    "\n",
    "Given two items $i$ and $j$, represented as rating vectors across all users $\\mathbf{r}_i, \\mathbf{r}_j$:\n",
    "$$\\text{similarity}(i, j) = \\cos(\\mathbf{r}_i, \\mathbf{r}_j) = \\frac{\\sum_u r_{u,i} \\cdot r_{u,j}}{\\sqrt{\\sum_u r_{u,i}^2} \\cdot \\sqrt{\\sum_u r_{u,j}^2}}$$\n",
    "- $1.0$ indicates identical co-rating profiles.\n",
    "- $0.0$ indicates completely orthogonal (no shared users or zero overlap).\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "40b35511",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load pre-trained model or train pipeline\n",
    "model = MovieRecommender.load(\"../models/recommender_model.joblib\")\n",
    "print(f\"Loaded model with {len(model.movie_to_idx):,} movies and {len(model.user_to_idx):,} users.\")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "b4f06a27",
   "metadata": {},
   "source": [
    "## 2. Finding Movies Similar to a Target Movie"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "efa5acab",
   "metadata": {},
   "outputs": [],
   "source": [
    "def show_similar(movie_title, top_k=5):\n",
    "    matches = model.search_movies(movie_title, top_k=1)\n",
    "    if not matches:\n",
    "        print(\"Movie not found!\")\n",
    "        return\n",
    "    m_id = matches[0][\"movieId\"]\n",
    "    print(f\"🎯 Target Movie: {matches[0]['title']} ({matches[0]['genres']})\")\n",
    "    sims = model.recommend_similar_movies(m_id, top_k=top_k)\n",
    "    return pd.DataFrame(sims)\n",
    "\n",
    "display(show_similar(\"Toy Story\", top_k=5))\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "36eefe5c",
   "metadata": {},
   "outputs": [],
   "source": [
    "display(show_similar(\"Matrix, The\", top_k=5))\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1235ea04",
   "metadata": {},
   "source": [
    "## 3. Generating Personalized Recommendations for a User\n",
    "We predict ratings for unrated movies and strictly exclude already-watched films.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "0a60fd2f",
   "metadata": {},
   "outputs": [],
   "source": [
    "user_id = 1\n",
    "user_watched = model.user_watched.get(user_id, set())\n",
    "print(f\"User {user_id} has watched {len(user_watched)} movies in training set.\")\n",
    "\n",
    "recs = model.recommend_item_based(user_id=user_id, top_k=5)\n",
    "rec_df = pd.DataFrame(recs)\n",
    "display(rec_df)\n",
    "\n",
    "# Check exclusion of watched movies\n",
    "overlaps = set(rec_df[\"movieId\"]).intersection(user_watched)\n",
    "print(f\"Watched movies in recommendation list: {len(overlaps)} (Must be 0)\")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "7cf453a6",
   "metadata": {},
   "source": [
    "## 4. Beginner Exercises\n",
    "1. Pick your favorite movie from the catalog. What are its top 5 most similar movies according to the cosine similarity matrix?\n",
    "2. Why is Item-Based Collaborative Filtering generally more stable over time than User-Based CF in large streaming systems like Netflix?\n",
    "   *(Hint: Item profiles change slowly as new ratings come in; user tastes change dynamically.)*\n"
   ]
  }
 ],
 "metadata": {},
 "nbformat": 4,
 "nbformat_minor": 5
}